# Testing Astroquery with JPL Horizons

This notebook makes a small observer-based ephemerides request to JPL Horizons, inspects the returned table, and plots right ascension and declination over time. The query requires internet access and may take a few seconds.

## 1. Install and Import Required Libraries

If imports fail because a package is missing, run the optional installation cell below, then restart the kernel if needed. `%pip` installs into the current notebook kernel.

In [1]:
# Optional: uncomment and run only if one or more imports below are unavailable.
# %pip install astroquery astropy matplotlib

In [2]:
from astroquery.jplhorizons import Horizons
import astropy
from astropy.time import Time
import matplotlib.pyplot as plt
import numpy as np

print(f"Astroquery {__import__('astroquery').__version__}; Astropy {astropy.__version__}")

Astroquery 0.4.6; Astropy 5.2.1


## 2. Set Up JPL Horizons Query Parameters

This example requests Mars as seen from Mauna Kea Observatory (IAU code `568`). The UTC interval is intentionally short; adjust the dates or target as needed. Horizons accepts target names as well as numeric IDs, though ambiguous names may require a more specific designation.

In [3]:
target = "Mars"
observer_location = "568"  # Mauna Kea Observatory
start_utc = "2026-10-26"
stop_utc = "2026-10-29"
step = "1d"

query_parameters = {
    "id": target,
    "location": observer_location,
    "epochs": {"start": start_utc, "stop": stop_utc, "step": step},
}
query_parameters

{'id': 'Mars',
 'location': '568',
 'epochs': {'start': '2026-10-26', 'stop': '2026-10-29', 'step': '1d'}}

## 3. Query Ephemerides Data from JPL Horizons

`Horizons(...).ephemerides()` returns an Astropy table of apparent observer ephemerides. If this request fails, confirm internet access and that the target, observatory code, and date range are accepted by Horizons.

In [4]:
horizons_query = Horizons(**query_parameters)
ephemerides = horizons_query.ephemerides()

print(f"Rows returned: {len(ephemerides)}")
print(f"Columns: {ephemerides.colnames}")

ValueError: Query failed without known error message; received the following response:
API VERSION: 1.2
API SOURCE: NASA/JPL Horizons API

wldini(): missing required file LTKERNL

 ERROR in VLRDC: Var not declared: IP_ADDR


## 4. Parse and Display Ephemerides Results

The result includes many columns; the selection below shows the observation time, apparent sky coordinates, observer-target distance, and visual magnitude when those fields are available.

In [ ]:
preferred_columns = ["datetime_str", "RA", "DEC", "delta", "V"]
selected_columns = [name for name in preferred_columns if name in ephemerides.colnames]
ephemerides[selected_columns]

## 5. Plot Ephemerides Data

Plot right ascension and declination against UTC time. Right ascension is plotted in degrees; for longer intervals spanning 0° the curve may wrap, so unwrap it before plotting.

In [ ]:
times = Time(ephemerides["datetime_jd"], format="jd", scale="utc").datetime
ra_deg = np.asarray(ephemerides["RA"], dtype=float)
dec_deg = np.asarray(ephemerides["DEC"], dtype=float)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, constrained_layout=True)
axes[0].plot(times, ra_deg, marker="o")
axes[0].set_ylabel("Right ascension (deg)")
axes[0].grid(True, alpha=0.3)
axes[1].plot(times, dec_deg, marker="o", color="tab:orange")
axes[1].set_ylabel("Declination (deg)")
axes[1].set_xlabel("UTC")
axes[1].grid(True, alpha=0.3)
fig.suptitle(f"{target} ephemerides from Mauna Kea ({observer_location})")
plt.show()